In [10]:
import pandas as pd
import numpy as np
df=pd.read_pickle(f"../df24.pkl")
#y=df['glu']
y=df['glu']
X=df.drop(columns=['dm','glu','hba1c'])

In [11]:
X.shape

(4325, 27)

In [12]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.metrics import r2_score, mean_squared_error

from xgboost import XGBRegressor


# =========================
# 1. Train / Test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print(y.describe())


# =========================
# 2. Preprocessing
# =========================
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), [
            'age', 'chol', 'hdl', 'tg', 'ldl', 'sbp', 'wt', 'ht', 'wc', 'bmi',
            'wk_smk', 'wk_alc', 'wk_mvpa_work', 'wk_mvpa_play', 'wk_walk',
            'wk_sleep', 'stress', 'wk_break', 'wk_lunch',
            'wk_dinner', 'wk_veg1', 'wk_veg2', 'wk_fruit',
            'edu', 'income'
        ]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ['sex', 'job'])
    ]
)


# =========================
# 3. XGBoost Regressor 정의
# =========================
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)


# =========================
# 4. Pipeline 구성
# =========================
pipe = Pipeline([
    ("preprocess", preprocess),
    ("xgb", model)
])


# =========================
# 5. 모델 학습
# =========================
pipe.fit(X_train, y_train)


# =========================
# 6. 예측
# =========================
y_pred = pipe.predict(X_test)


# =========================
# 7. 회귀 성능 평가
# =========================
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")


count    4325.000000
mean      101.042081
std        20.427068
min        65.000000
25%        90.000000
50%        96.000000
75%       106.000000
max       349.000000
Name: glu, dtype: float64
R2 score: 0.0862
RMSE: 21.4041
